# 要約と集計

In [1]:
from datetime import date
from functools import lru_cache
from textblob import TextBlob
import os

import numpy as np
import polars as pl
import polars.selectors as cs
from scipy.special import softmax
from sklearn.preprocessing import StandardScaler

## 定数定義

In [2]:
DATA_PAR_PATH = os.path.join('..','..','data')
INPUT_CSV_PATH_TOP2000 = os.path.join(DATA_PAR_PATH,'top2000-2023.xlsx')
INPUT_CSV_PATH_SALES = os.path.join(DATA_PAR_PATH,'sales.csv')

## コンテキストによるグルーピング

In [3]:
top2000 = pl.read_excel(
    INPUT_CSV_PATH_TOP2000,
    read_options={'skip_rows': 1},  # 余計な空行をスキップ
    engine='calamine'
).set_sorted('positie')

`.set_sorted()`によって、polars側にソートの「高速パス最適化」を適用できる。  
これで指定したカラムはソートフラグがつくため、後からソート処理をする際に指定したカラムに関する処理をスキップできる。その点で高速化できる

In [4]:
(
    top2000
    .group_by('jaar')
    .agg(
        (
            pl.concat_str(
                pl.col('artiest'),
                pl.lit(' - '),
                pl.col('titel')
            )
        ).alias('songs')
    )
    .sort('jaar', descending=True)
)

jaar,songs
i64,list[str]
2022,"[""Son Mieux - Multicolor"", ""Bankzitters - Je Blik Richting Mij"", … ""Måneskin - THE LONELIEST""]"
2021,"[""Goldband - Noodgeval"", ""Bankzitters - Stapelgek"", … ""Olivia Rodrigo - Drivers License""]"
2020,"[""DI-RECT - Soldier On"", ""Miss Montreal - Door De Wind"", … ""Dua Lipa ft. DaBaby - Levitating""]"
2019,"[""Danny Vera - Roller Coaster"", ""Floor Jansen & Henk Poort - Phantom Of The Opera"", … ""Tino Martin - Zij Weet Het""]"
2018,"[""Lady Gaga & Bradley Cooper - Shallow"", ""White Lies - Time To Give"", … ""Calvin Harris & Dua Lipa - One Kiss""]"
…,…
1960,"[""Etta James - At Last"", ""Shadows - Apache""]"
1959,"[""Jacques Brel - Ne Me Quitte Pas"", ""Elvis Presley - Hound Dog""]"
1958,"[""Chuck Berry - Johnny B. Goode"", ""Ella Fitzgerald & Louis Armstrong - Summertime""]"


`.group_by()`した内容を利用する場合は、`.agg()`で列を追加するしかない。  
`.with_columns()`は`.group_by()`の後に使えない

In [5]:
(
    top2000
    .group_by('jaar', maintain_order=True)
    .head(3)
    .sort('jaar', descending=True)
    .head(9)
)

jaar,positie,titel,artiest
i64,i64,str,str
2022,179,"""Multicolor""","""Son Mieux"""
2022,370,"""Je Blik Richting Mij""","""Bankzitters"""
2022,395,"""L'enfer""","""Stromae"""
2021,55,"""Noodgeval""","""Goldband"""
2021,149,"""Stapelgek""","""Bankzitters"""
2021,210,"""Dat Heb Jij Gedaan""","""Meau"""
2020,19,"""Soldier On""","""DI-RECT"""
2020,38,"""Door De Wind""","""Miss Montreal"""
2020,77,"""Impossible (Orchestral Version…","""Nothing But Thieves"""


`maintain_order=True`をつけることによって、polarsで内部的に並列処理されている内容の順序を保持できる。このパラメータがないと、並列処理の都合上、バラバラに出力される可能性がある

In [6]:
(
    top2000
    .group_by('jaar', maintain_order=True)
    .tail(3)
    .sort('jaar', descending=True)
    .head(9)
)

jaar,positie,titel,artiest
i64,i64,str,str
2022,1391,"""De Diepte""","""S10"""
2022,1688,"""Zeit""","""Rammstein"""
2022,1716,"""THE LONELIEST""","""Måneskin"""
2021,1865,"""Bon Gepakt""","""Donnie & Rene Froger"""
2021,1978,"""Hold On""","""Armin van Buuren ft. Davina Mi…"
2021,2000,"""Drivers License""","""Olivia Rodrigo"""
2020,1824,"""Smoorverliefd""","""Snelle"""
2020,1879,"""The Business""","""Tiësto"""
2020,1902,"""Levitating""","""Dua Lipa ft. DaBaby"""


`head(1) = first()`、`tail(1) = last()`と同義

In [7]:
(
    top2000
    .group_by('artiest')
    .len()
    .sort('len', descending=True)
    .head(10)
)

artiest,len
str,u32
"""Queen""",34
"""The Beatles""",31
"""ABBA""",25
"""Bruce Springsteen""",22
"""The Rolling Stones""",22
"""Coldplay""",20
"""Fleetwood Mac""",20
"""Michael Jackson""",20
"""David Bowie""",18


In [8]:
df = pl.read_csv(INPUT_CSV_PATH_SALES)
df.columns

['Date',
 'Day',
 'Month',
 'Year',
 'Customer_Age',
 'Age_Group',
 'Customer_Gender',
 'Country',
 'State',
 'Product_Category',
 'Sub_Category',
 'Product',
 'Order_Quantity',
 'Unit_Cost',
 'Unit_Price',
 'Profit',
 'Cost',
 'Revenue']

In [9]:
(
    df
    .select('Product_Category', 'Sub_Category', 'Unit_Price')
    .group_by('Product_Category', 'Sub_Category')
    .max()
    .sort('Unit_Price', descending=True)
    .head(10)
)

Product_Category,Sub_Category,Unit_Price
str,str,i64
"""Bikes""","""Road Bikes""",3578
"""Bikes""","""Mountain Bikes""",3400
"""Clothing""","""Vests""",2384
"""Bikes""","""Touring Bikes""",2384
"""Accessories""","""Bike Stands""",159
"""Accessories""","""Bike Racks""",120
"""Clothing""","""Shorts""",70
"""Clothing""","""Socks""",70
"""Accessories""","""Hydration Packs""",55


`.group_by()`って一括でグルーピングできるのか

In [10]:
(
    df
    .select('Country', 'Profit')
    .group_by('Country')
    .sum()
    .sort('Profit', descending=True)
)

Country,Profit
str,i64
"""United States""",11073644
"""Australia""",6776030
"""United Kingdom""",4413853
"""Canada""",3717296
"""Germany""",3359995
"""France""",2880282


In [11]:
(
    df
    .select('Sub_Category', 'Product')
    .group_by('Sub_Category')
    .n_unique()
    .sort('Product', descending=True)
    .head(10)
)

Sub_Category,Product
str,u32
"""Road Bikes""",38
"""Mountain Bikes""",28
"""Touring Bikes""",22
"""Tires and Tubes""",11
"""Jerseys""",8
"""Gloves""",4
"""Vests""",4
"""Bottles and Cages""",3
"""Shorts""",3


`.n_unique()`を使って、ユニーク数のカウンティングが可能

In [12]:
(
    df
    .select('Age_Group', 'Order_Quantity')
    .group_by('Age_Group')
    .mean()
    .sort('Order_Quantity', descending=True)
)

Age_Group,Order_Quantity
str,f64
"""Seniors (64+)""",13.530137
"""Youth (<25)""",12.124018
"""Adults (35-64)""",12.045303
"""Young Adults (25-34)""",11.560899


In [13]:
(
    df
    .select('Age_Group', 'Revenue')
    .group_by('Age_Group')
    .quantile(0.9)
    .sort('Revenue', descending=True)
)

Age_Group,Revenue
str,f64
"""Young Adults (25-34)""",2227.0
"""Adults (35-64)""",2217.0
"""Youth (<25)""",1997.0
"""Seniors (64+)""",943.0


`.quantile()`を使って、上位k%値を取得できる

## 応用

In [14]:
(
    df
    .select('Country', 'Profit', 'Revenue')
    .group_by('Country')
    .agg(
        pl.col('Profit'),
        pl.col('Revenue')
    )
)

Country,Profit,Revenue
str,list[i64],list[i64]
"""Canada""","[590, 590, … 630]","[950, 950, … 1014]"
"""Australia""","[1366, 1188, … 655]","[2401, 2088, … 1183]"
"""United States""","[524, 407, … 542]","[929, 722, … 878]"
"""France""","[427, 427, … 655]","[787, 787, … 1207]"
"""Germany""","[160, 53, … 746]","[295, 98, … 1250]"
"""United Kingdom""","[1053, 1053, … 112]","[1728, 1728, … 184]"


Countryにてグルーピングした内容に対して、それらの結果をlist型として集計している。  
さっきまではグルーピングした内容のソートや計算だけだったけれど、集計もできるのか

In [15]:
(
    df
    .select('Country', 'Profit', 'Revenue')
    .group_by('Country')
    .agg(
        pl.col('Profit').alias('All Profits Per Transactions'),
        pl.col('Revenue').name.prefix('All ')
    )
)

Country,All Profits Per Transactions,All Revenue
str,list[i64],list[i64]
"""Canada""","[590, 590, … 630]","[950, 950, … 1014]"
"""United Kingdom""","[1053, 1053, … 112]","[1728, 1728, … 184]"
"""United States""","[524, 407, … 542]","[929, 722, … 878]"
"""Australia""","[1366, 1188, … 655]","[2401, 2088, … 1183]"
"""Germany""","[160, 53, … 746]","[295, 98, … 1250]"
"""France""","[427, 427, … 655]","[787, 787, … 1207]"


列名を操作するときは`.str`じゃなくて`.name`だったか。忘れてた

In [16]:
(
    df
    .select('Country', 'Profit', 'Revenue')
    .group_by('Country')
    .agg(
        pl.col('Profit').sum().alias('Total Profit'),
        pl.col('Profit').mean().alias('Average Profit per Transactions'),
        pl.col('Revenue').sum().alias('Total Revenue'),
        pl.col('Revenue').mean().alias('Averate Revenue per Transactions')
    )
)

Country,Total Profit,Average Profit per Transactions,Total Revenue,Averate Revenue per Transactions
str,i64,f64,i64,f64
"""France""",2880282,261.891435,8432872,766.764139
"""Australia""",6776030,283.089489,21302059,889.959016
"""Canada""",3717296,262.187615,7935738,559.721964
"""Germany""",3359995,302.756803,8978596,809.028293
"""United Kingdom""",4413853,324.071439,10646196,781.659031
"""United States""",11073644,282.447687,27975547,713.552696


In [17]:
(
    df
    .select('Country', 'Profit', 'Revenue')
    .group_by('Country')
    .agg(
        pl.all().sum().name.prefix('Total '),
        pl.all().mean().name.prefix('Averate ')
    )
)

Country,Total Profit,Total Revenue,Averate Profit,Averate Revenue
str,i64,i64,f64,f64
"""United States""",11073644,27975547,282.447687,713.552696
"""France""",2880282,8432872,261.891435,766.764139
"""United Kingdom""",4413853,10646196,324.071439,781.659031
"""Germany""",3359995,8978596,302.756803,809.028293
"""Australia""",6776030,21302059,283.089489,889.959016
"""Canada""",3717296,7935738,262.187615,559.721964


In [18]:
(
    df
    .select('Country', 'Profit')
    .group_by('Country')
    .agg(
        (pl.col('Profit') > 1000)
        .alias('Profit > 1000'),
        (pl.col('Profit') > 1000)
        .sum()  # 1000より大きい値の合計値
        .alias('Number of Transactions with Profit > 1000')
    )
)

Country,Profit > 1000,Number of Transactions with Profit > 1000
str,list[bool],u32
"""Australia""","[true, true, … false]",1233
"""France""","[false, false, … false]",482
"""United Kingdom""","[true, true, … false]",788
"""United States""","[false, false, … false]",2623
"""Germany""","[false, false, … false]",659
"""Canada""","[false, false, … false]",868


In [19]:
def custom_agg(column: str) -> pl.Expr:
    return (column > 1000).alias('Profit > 1000'), (column > 1000).sum().alias('Number of Transactions with Profit > 1000')

(
    df
    .select('Country', 'Profit')
    .group_by('Country')
    .agg(
        custom_agg(pl.col('Profit'))
    )
)

Country,Profit > 1000,Number of Transactions with Profit > 1000
str,list[bool],u32
"""United States""","[false, false, … false]",2623
"""France""","[false, false, … false]",482
"""Australia""","[true, true, … false]",1233
"""Canada""","[false, false, … false]",868
"""United Kingdom""","[true, true, … false]",788
"""Germany""","[false, false, … false]",659


`.agg()`の中に関数を入れることが可能なのか。  
にしても、`pl.col()`って型としてはstring型なのね

## ユーザー定義関数

In [20]:
def analyze_sentiment(review):
    return TextBlob(review).sentiment.polarity

df = pl.DataFrame({
    'reviews': [
        'This product is great!',
        'Terrible service',
        'Okey, but not what I expected',
        'Excellent! I love it.'
    ]
})

df = df.with_columns(
    pl.col('reviews')
    .map_elements(
        analyze_sentiment,
        return_dtype=pl.Float64
    )
    .alias('sentiment_score')
)

df

reviews,sentiment_score
str,f64
"""This product is great!""",1.0
"""Terrible service""",-1.0
"""Okey, but not what I expected""",-0.1
"""Excellent! I love it.""",0.75


`TextBlob`で感情分析ができるのか。このライブラリ初めて知ったけれど、手軽でいいね。

ユーザーが定義した関数を実行する際は、polarsがRust製だからといえどpython環境で実行される点で、実行速度はpolarsで用意されている処理と比べると低速になる。  
特に、pythonはGILによって並列処理の速度向上が望めない。ただし、python3.13ではGILの設定をいじれるようになったはずなので、python3.13になるとpolarsとの相性も多少よくなる可能性があるかも？

In [21]:
df = pl.DataFrame({
    'x': [1, 2, 3, 4]
})

def add_one(x):
    return x + 1

df.with_columns(
    pl.col('x')
    .map_elements(
        add_one,
        return_dtype=pl.Int64
    )
    .alias('x + 1')
)

/var/folders/hm/1554tfbj14b8r1wpst4sml2h0000gn/T/ipykernel_12358/2847855123.py:10: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("x").map_elements(add_one)
with this one instead:
  + pl.col("x") + 1

  .map_elements(


x,x + 1
i64,i64
1,2
2,3
3,4
4,5


低速なpythonよりもpolarsで代替できる場合は、warinigでお知らせしてくれる。なんて親切

In [22]:
df = pl.DataFrame({
    'x': [1, 1, 3, 3]
})

@lru_cache(maxsize=None)
def add_one(x):
    return x + 1

df.with_columns(
    pl.col('x')
    .map_elements(
        add_one,
        return_dtype=pl.Int64
    )
    .alias('x + 1')
)

x,x + 1
i64,i64
1,2
1,2
3,4
3,4


`@lru_cache`デコレータを使うことで、効率的にメモリを使えるので、速度面での向上が図れる

In [23]:
df = pl.DataFrame({
    'feature1': [0.3, 0.2, 0.4, 0.1, 0.2, 0.3, 0.5],
    'feature2': [32, 50, 70, 65, 0, 10, 15],
    'label': [1, 0, 1, 0, 1, 0, 0]
})

result = df.select(
    'label',
    cs.starts_with('feature').map_batches(
        lambda x: softmax(x.to_numpy())
    )
)

result

label,feature1,feature2
i64,f64,f64
1,0.143782,3.1181e-17
0,0.130099,2.0474e-9
1,0.158904,0.993307
0,0.117719,0.006693
1,0.130099,3.9488e-31
0,0.143782,8.6979e-27
0,0.175616,1.2909e-24


`.map_batches()`では、スカラー（numpy配列とか）に対する処理を記述できる。  
今回の例では、`feature1 / feature2`の各要素をnumpy配列に変化して、softmax関数に入れる操作を実施している

In [24]:
def scale_temperature(group):
    scaler = StandardScaler()
    scaled_values = scaler.fit_transform(group[['temperature']].to_numpy())

    return group.with_columns(pl.Series(values=scaled_values.flatten(), name='scaled_feature'))

df = pl.DataFrame({
    'group': ['USA', 'USA', 'USA', 'USA', 'NL', 'NL', 'NL'],
    'temperature' : [32, 50, 70, 65, 0, 10, 15]
})

result = df.group_by('group').map_groups(scale_temperature)
result

group,temperature,scaled_feature
str,i64,f64
"""NL""",0,-1.336306
"""NL""",10,0.267261
"""NL""",15,1.069045
"""USA""",32,-1.502872
"""USA""",50,-0.287066
"""USA""",70,1.063831
"""USA""",65,0.726107


`.map_group()`を使うことで、グルーピングしたデータに対してpythonでの関数を適用できる

In [25]:
df = pl.DataFrame({
    'group': ['USA', 'USA', 'USA', 'USA', 'NL', 'NL', 'NL'],
    'temperature' : [32, 50, 70, 65, 0, 10, 15]
})

for group in df.group_by(['group']):
    print(group)

(('USA',), shape: (4, 2)
┌───────┬─────────────┐
│ group ┆ temperature │
│ ---   ┆ ---         │
│ str   ┆ i64         │
╞═══════╪═════════════╡
│ USA   ┆ 32          │
│ USA   ┆ 50          │
│ USA   ┆ 70          │
│ USA   ┆ 65          │
└───────┴─────────────┘)
(('NL',), shape: (3, 2)
┌───────┬─────────────┐
│ group ┆ temperature │
│ ---   ┆ ---         │
│ str   ┆ i64         │
╞═══════╪═════════════╡
│ NL    ┆ 0           │
│ NL    ┆ 10          │
│ NL    ┆ 15          │
└───────┴─────────────┘)


## reduceとfold

In [26]:
df = pl.DataFrame({
    'col1': [2],
    'col2': [3],
    'col3': [4]
})

df.with_columns(
    pl.fold(
        acc=pl.lit(0),
        function=lambda acc, x: acc + x,
        exprs=pl.col('*')  # ワイルドカード
    ).alias('sum')
)

col1,col2,col3,sum
i64,i64,i64,i64
2,3,4,9


`exprs=`のパラメータの意味が理解できていない。あとで詰める

In [27]:
df = pl.DataFrame({
    'product_A': [10, 20, 30],
    'product_B': [20, 30, 40],
    'product_C': [30, 40, 50]
})

weights = {
    'product_A': 0.5,
    'product_B': 1.5,
    'product_C': 2.0
}

weighted_exprs = [
    (pl.col(product) * weight).alias(product)
    for product, weight in weights.items()
]

df_with_weighted_sum = df.with_columns(
    pl.fold(
        acc=pl.lit(0),
        function=lambda acc, x: acc + x,
        exprs=weighted_exprs
    ).alias('weighted_sum')
)

df_with_weighted_sum

product_A,product_B,product_C,weighted_sum
i64,i64,i64,f64
10,20,30,95.0
20,30,40,135.0
30,40,50,175.0


`exprs=`パラメータの意味が理解できた。  
ここでpolarsの式を定義してあげると、各列に対して重み付けが可能になる。仮に重みが必要ない場合であったとしても、`pl.fold()`ではこのパラメータが必須なので、その時にワイルドカードを使っていたのか

`reduce`と`fold`は基本的な操作が同じであるが、`reduce`は累積の初期値が初めに見つかった値であるのに対して、`fold`は`acc`パラメータで初期値を設定できる。というだけの違い

## over

In [28]:
(top2000
    .select(
        'jaar',
        'artiest',
        'titel',
        'positie',
        pl.col('positie')
        .rank()
        .over('jaar')
        .alias('year_rank')
    )
    .sample(10, seed=42)
)

jaar,artiest,titel,positie,year_rank
i64,str,str,i64,f64
2013,"""Stromae""","""Papaoutai""",318,6.0
1969,"""John Denver""","""Leaving On A Jet Plane""",607,16.0
1971,"""Led Zeppelin""","""Immigrant Song""",590,19.0
2009,"""Anouk""","""For Bitter Or Worse""",1453,23.0
2015,"""Snollebollekes""","""Links Rechts""",1076,14.0
1984,"""Alphaville""","""Forever Young""",302,11.0
1977,"""ABBA""","""Take A Chance On Me""",636,23.0
1975,"""Rod Stewart""","""Sailing""",918,20.0
1986,"""Metallica""","""Master Of Puppets""",29,1.0


`.over()`を使うことで、指定した列によるグルーピングをしつつ、元のテーブル構造を崩すことなく出力できる。  
`.group_by()`をして算出することも可能だが、その場合はテーブル構造全体がグルーピングされてしまうので、それをせずに操作したい時に`.over()`の出番ってわけか

## rolling
`group_by_dynamic`は固定長サイズでの作成だったのに対して、`rolling`では移動平均といった可変長の計算を実行するのに便利

In [29]:
dates = pl.date_range(
    start=date(2024, 4, 1),
    end=date(2024, 4, 14),
    interval='1d',
    eager=True
)

dates = dates.filter(dates.dt.weekday() < 6)
dates_repeated = pl.concat([dates, dates]).sort()

df = pl.DataFrame({
    'date': dates_repeated,
    'store': ['Store A', 'Store B'] * dates.len(),
    'sales': [
        200, 150, 220, 160, 250, 180, 270, 190, 280, 210,
        210, 170, 220, 180, 240, 190, 250, 200, 260, 210
    ]
}).set_sorted('date').set_sorted('store')

`.set_sorted()`で列を指定する際は、1個ずつ指定する必要がある（バージョンが上がる前までは複数指定ができてたっぽい？）

In [30]:
df

date,store,sales
date,str,i64
2024-04-01,"""Store A""",200
2024-04-01,"""Store B""",150
2024-04-02,"""Store A""",220
2024-04-02,"""Store B""",160
2024-04-03,"""Store A""",250
…,…,…
2024-04-10,"""Store B""",190
2024-04-11,"""Store A""",250
2024-04-11,"""Store B""",200


In [31]:
result = (
    df.rolling(
        index_column='date',
        period='7d',
        group_by='store'
    ).agg(
        pl.sum('sales').alias('sum_of_last_7_days_sales')
    )
)

final_df = df.join(result, on=['date', 'store'])

final_df

date,store,sales,sum_of_last_7_days_sales
date,str,i64,i64
2024-04-01,"""Store A""",200,200
2024-04-02,"""Store A""",220,420
2024-04-03,"""Store A""",250,670
2024-04-04,"""Store A""",270,940
2024-04-05,"""Store A""",280,1220
…,…,…,…
2024-04-08,"""Store B""",170,910
2024-04-09,"""Store B""",180,930
2024-04-10,"""Store B""",190,940


`.rolling()`によって、自身のデータから7日前までの合計値を算出している。  
なるほど。`.rolling()`で時間情報をベースにどれくらいの範囲に対して影響を与えるかを指定して、その集計結果を`.agg()`によってテーブルに挿入しているのか。複雑だけれど、慣れたら出番ありそう